 # Application Configuration

 This module centralizes the application's configuration.

 It loads values from environment variables, validates them with Pydantic,
 applies safe defaults, and exposes a cached settings object.

In [ ]:
import os
from functools import lru_cache
from dotenv import load_dotenv
from pydantic import BaseModel, Field, PostgresDsn
from typing import Optional

 ## Design Goals

 The configuration module has four main responsibilities:

 1. Load configuration values from the `.env` file and the system environment.
 2. Convert raw strings into the correct Python types.
 3. Validate structured values such as the PostgreSQL database URL.
 4. Cache the final settings object so it is created only once.

 This keeps configuration logic in one place and prevents environment variables
 from being accessed throughout the application.
 `Settings` defines the complete configuration schema for the application.

 Pydantic validates the values when the object is created. For example,
 `database_url` must contain a valid PostgreSQL connection URL.

 Optional fields default to `None`. This allows the application to start
 even when optional services, such as SMTP, are not configured.

 Default values provide safe development settings when an environment variable
 is missing.

In [ ]:
class Settings(BaseModel):
    """Application configurations loaded from .env."""
    app_name: str = Field(default="reactive_agent", description="Application name")
    environment: str = Field(default="development", description="Deployment environment")
    debug: bool = Field(default=False, description="Enable debug mode")
    log_level: str = Field(default="INFO", description="Default logging level")
    database_url: Optional[PostgresDsn] = Field(default=None, description="Async PostgreSQL connection URL (postgresql+asyncpg://...)")
    llm_generator: str = Field(default="groq/compound", description="Model used for agent generation")
    llm_clarifier: str = Field(default="groq/compound", description="Model used to clarify user requests and planning")
    cors_origins: list[str] = Field(default_factory=lambda: ["http://localhost:3000"])
    smtp_host: Optional[str] = Field(default=None, description="SMTP host for email sending")
    smtp_port: Optional[int] = Field(default=None, description="SMTP port for sending email")
    smtp_user: Optional[str] = Field(default=None, description="SMTP username")
    smtp_password: Optional[str] = Field(default=None, description="SMTP password")
    smtp_from_email: Optional[str] = Field(default=None, description="Default email sender")

    @property
    def smtp_enabled(self) -> bool:
        return bool(
            self.smtp_host
            and self.smtp_port
            and self.smtp_user
            and self.smtp_password
            and self.smtp_from_email
        )

 ## SMTP Availability

 `smtp_enabled` is a computed property.

 It returns `True` only when all required SMTP settings are available.
 The application can use this flag to decide whether email functionality
 should be enabled.

 This avoids starting an email connection when the SMTP configuration
 is incomplete.

 ## Environment Variable Parsing

 Environment variables are always loaded as strings.

 The helper functions convert these raw values into the types expected
 by the `Settings` model:

 - `_parse_bool()` converts text into a Boolean value.
 - `_parse_int()` converts text into an integer safely.
 - `_parse_origins()` converts a comma-separated string into a list.

 Invalid or empty values are handled defensively instead of causing
 an unexpected application crash during startup.

In [ ]:
def _parse_bool(value: str, default: bool = False) -> bool:
    if not value:
        return default
    return value.strip().lower() in {"1","true","yes","on",}

def _parse_int(value: Optional[str]) -> Optional[int]:
    if not value or not value.strip():
        return None

    try:
        return int(value.strip())
    except ValueError:
        return None


def _parse_origins(value: Optional[str]) -> list[str]:
    if not value:
        return ["http://localhost:3000"]

    return [
        origin.strip()
        for origin in value.split(",")
        if origin.strip()
    ]

 ## Loading the Configuration

 `get_settings()` is the single entry point for accessing application settings.

 `load_dotenv()` loads values from the `.env` file without replacing
 environment variables that are already defined by the operating system.

 The values are read, converted, and passed to the `Settings` model,
 where Pydantic performs the final validation.

In [ ]:
@lru_cache
def get_settings() -> Settings:
    load_dotenv()

    smtp_port_raw = os.getenv("SMTP_PORT")

    return Settings(
        app_name=os.getenv("APP_NAME", "reactive_agent"),
        environment=os.getenv("ENVIRONMENT", "development"),
        debug=_parse_bool(os.getenv("DEBUG", "False")),
        log_level=os.getenv("LOG_LEVEL", "INFO"),
        database_url=os.getenv("DATABASE_URL") or None,
        llm_generator=os.getenv("LLM_GENERATOR","groq/compound",),
        llm_clarifier=os.getenv("LLM_CLARIFIER","groq/compound",),
        cors_origins=_parse_origins(os.getenv("CORS_ORIGINS")),
        smtp_host=os.getenv("SMTP_HOST") or None,
        smtp_port=_parse_int(smtp_port_raw),
        smtp_user=os.getenv("SMTP_USER") or None,
        smtp_password=os.getenv("SMTP_PASSWORD") or None,
        smtp_from_email=os.getenv("SMTP_FROM_EMAIL") or None,
    )

 ## Why Use `@lru_cache`?

 `@lru_cache` stores the result of the first call to `get_settings()`.

 Subsequent calls return the same configuration object instead of loading
 the `.env` file and rebuilding the `Settings` model again.

 This provides a simple way to reuse one application-wide settings object
 without introducing a global variable.

 ## Usage

 Other modules should access configuration through `get_settings()`.

 This keeps environment-variable access isolated inside the configuration
 module.
 Store only non-sensitive examples in `.env.example`.
 Passwords, API keys, and database credentials should be provided through
 environment variables or a dedicated secrets manager.